# Etapa 2: Aprendizaje Profundo Supervisado (Deep Learning)
**Materia:** Machine Learning y Ciencia de los Datos (2026)  
**Docente:** Lic. Patricia Rodriguez Bilbao  
**Area Asignada:** Seguridad | **Dataset:** PhiUSIIL Phishing URL Dataset  

---

## Objetivos de la Practica:
1. Cargar los datos preprocesados con estandarizacion Z-Score.
2. Construir una arquitectura de Red Neuronal Profunda (MLP) con regularizacion (BatchNormalization y Dropout).
3. Compilar el modelo con Binary Crossentropy y optimizador Adam.
4. Entrenar registrando curvas de perdida y precision en Train y Validacion.
5. Evaluar en el conjunto de prueba independiente (Matriz de Confusion, ROC-AUC, Precision, Recall y F1-Score).
6. Serializar el modelo entrenado en formato .keras dentro de `models/deep_learning/guardados/`.

In [ ]:
import os
import sys

# Configurar ruta raiz del proyecto dinamicamente
current_dir = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(current_dir, 'requirements.txt')) and os.path.dirname(current_dir) != current_dir:
    current_dir = os.path.dirname(current_dir)
ROOT_DIR = current_dir
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from src.data_loader import load_phiusiil_processed, load_phiusiil_sample

MODELS_DIR = os.path.join(ROOT_DIR, 'models', 'deep_learning', 'guardados')
RESULTS_DIR = os.path.join(ROOT_DIR, 'models', 'deep_learning', 'resultados')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('[OK] TensorFlow Version:', tf.__version__)

## 1. Carga de Conjuntos de Entrenamiento y Prueba

In [ ]:
# Cargar datos procesados (Z-Score aplicado)
train_df, test_df = load_phiusiil_processed()

TARGET = 'label'
EXCLUDE_COLS = [TARGET, 'URL']
FEATURE_COLS = [col for col in train_df.select_dtypes(include=[np.number]).columns if col not in EXCLUDE_COLS]

X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET].values

X_test = test_df[FEATURE_COLS].values
y_test = test_df[TARGET].values

print(f'[INFO] Matriz X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'[INFO] Matriz X_test:  {X_test.shape}, y_test:  {y_test.shape}')

## 2. Definicion de la Arquitectura de la Red Neuronal (MLP)
Se implementa una red neuronal profunda con regularizacion `Dropout` y `BatchNormalization` para optimizar el entrenamiento.

In [ ]:
def build_mlp_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

model = build_mlp_model(X_train.shape[1])
model.summary()

## 3. Entrenamiento con Validacion y Early Stopping

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

## 4. Curvas de Aprendizaje (Loss y Accuracy)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Curva de Perdida
ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
ax1.set_title('Funcion de Perdida (Binary Crossentropy)', fontweight='bold')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Loss')
ax1.legend()

# Curva de Precision
ax2.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
ax2.set_title('Precision (Accuracy)', fontweight='bold')
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Accuracy')
ax2.legend()

curva_path = os.path.join(RESULTS_DIR, '01_curvas_entrenamiento_mlp.png')
plt.savefig(curva_path, dpi=300)
plt.show()
print(f'[OK] Grafica guardada en: {curva_path}')

## 5. Evaluacion en Conjunto de Prueba y Matriz de Confusion

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs >= 0.5).astype(int).ravel()

print('Reporte de Clasificacion en Test:')
print(classification_report(y_test, y_pred, target_names=['Phishing (0)', 'Legitimo (1)']))

auc_score = roc_auc_score(y_test, y_pred_probs)
print(f'Area bajo la Curva ROC (AUC): {auc_score:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=['Phishing', 'Legitimo'], yticklabels=['Phishing', 'Legitimo'])
plt.title('Matriz de Confusion (Test Set)', fontweight='bold')
plt.xlabel('Prediccion')
plt.ylabel('Valor Real')
cm_path = os.path.join(RESULTS_DIR, '02_matriz_confusion_mlp.png')
plt.savefig(cm_path, dpi=300)
plt.show()

## 6. Serializacion del Modelo Entrenado

In [ ]:
model_save_path = os.path.join(MODELS_DIR, 'modelo_mlp_phishing.keras')
model.save(model_save_path)
print(f'[OK] Modelo serializado en: {model_save_path}')